In [ ]:
Nama:Fahmiori Afnil
NIM:2411513033
TUGAS PEMPAR 4

In [8]:
import asyncio
import time
import aiohttp
import matplotlib.pyplot as plt

# 20-30 nama paket PyPI valid
daftar_paket_banyak = [
    "requests", "numpy", "pandas", "scipy", "matplotlib",
    "flask", "django", "fastapi", "torch", "tensorflow",
    "scikit-learn", "pillow", "urllib3", "pydantic", "pytest",
    "black", "boto3", "click", "jinja2", "sqlalchemy",
    "sympy", "seaborn", "plotly", "beautifulsoup4", "wheel"
]

async def fetch_package(session, package_name, semaphore):
    async with semaphore:
        url = f"https://pypi.org/pypi/{package_name}/json"
        try:
            async with session.get(url) as response:
                return await response.json()
        except Exception as e:
            return None

async def run_experiment(limit):
    semaphore = asyncio.Semaphore(limit)
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_package(session, pkg, semaphore) for pkg in daftar_paket_banyak]
        start_time = time.perf_counter()
        await asyncio.gather(*tasks)
        end_time = time.perf_counter()
        return end_time - start_time

async def main_eksperimen_1():
    batas_konkurensi_list = [1, 3, 5, 10, 20]
    waktu_list = []

    print("--- Memulai Eksperimen Skala ---")
    for limit in batas_konkurensi_list:
        waktu = await run_experiment(limit)
        waktu_list.append(waktu)
        print(f"BATAS_KONKURENSI = {limit:2d} | Waktu Eksekusi: {waktu:.4f} detik")

    # Plot Waktu vs Batas Konkurensi
    plt.figure(figsize=(8, 5))
    plt.plot(batas_konkurensi_list, waktu_list, marker='o', color='b', linestyle='-')
    plt.title('Eksperimen Skala: Waktu Eksekusi vs Batas Konkurensi')
    plt.xlabel('BATAS_KONKURENSI')
    plt.ylabel('Waktu Total (detik)')
    plt.grid(True)
    plt.show()

# Jalankan di Jupyter Notebook
await main_eksperimen_1()

ModuleNotFoundError: No module named 'aiohttp'

In [ ]:
2. Deteksi Blocking Call
Kode ini mendemonstrasikan bagaimana fungsi blokir (blocking call) mengganggu eksekusi event loop tunggal pada asyncio.

In [6]:

import asyncio
import time

# Coroutine 1: Non-blocking sleep
async def task_asyncio_sleep():
    print("[Asyncio Sleep] Mulai...")
    await asyncio.sleep(2)
    print("[Asyncio Sleep] Selesai.")

# Coroutine 2: Blocking sleep (time.sleep)
async def task_time_sleep():
    print("[Time Sleep] Mulai (BLOCKING)...")
    time.sleep(2)  # Memblokir event loop
    print("[Time Sleep] Selesai.")

# Coroutine 3: CPU-bound murni (tanpa await)
async def task_cpu_bound():
    print("[CPU Bound] Mulai (BLOCKING CPU)...")
    # Hitung faktorial/operasi besar dalam loop
    total = 1
    for i in range(1, 15000000):
        total += i
    print("[CPU Bound] Selesai.")

# Coroutine Pengukur
async def task_pengukur():
    print("  --> [Pengukur] Mulai mencatat waktu...")
    start = time.perf_counter()
    for i in range(5):
        await asyncio.sleep(0.5)
        now = time.perf_counter() - start
        print(f"  --> [Pengukur] Detak ke-{i+1} pada {now:.2f} detik")

async def main_eksperimen_2():
    print("=== UJI 1: Non-Blocking (asyncio.sleep) + Pengukur ===")
    await asyncio.gather(task_asyncio_sleep(), task_pengukur())

    print("\n=== UJI 2: Blocking (time.sleep) + Pengukur ===")
    await asyncio.gather(task_time_sleep(), task_pengukur())

    print("\n=== UJI 3: Blocking CPU-bound + Pengukur ===")
    await asyncio.gather(task_cpu_bound(), task_pengukur())

await main_eksperimen_2()

=== UJI 1: Non-Blocking (asyncio.sleep) + Pengukur ===
[Asyncio Sleep] Mulai...
  --> [Pengukur] Mulai mencatat waktu...
  --> [Pengukur] Detak ke-1 pada 0.51 detik
  --> [Pengukur] Detak ke-2 pada 1.02 detik
  --> [Pengukur] Detak ke-3 pada 1.54 detik
[Asyncio Sleep] Selesai.
  --> [Pengukur] Detak ke-4 pada 2.05 detik
  --> [Pengukur] Detak ke-5 pada 2.56 detik

=== UJI 2: Blocking (time.sleep) + Pengukur ===
[Time Sleep] Mulai (BLOCKING)...
[Time Sleep] Selesai.
  --> [Pengukur] Mulai mencatat waktu...
  --> [Pengukur] Detak ke-1 pada 0.50 detik
  --> [Pengukur] Detak ke-2 pada 1.02 detik
  --> [Pengukur] Detak ke-3 pada 1.53 detik
  --> [Pengukur] Detak ke-4 pada 2.03 detik
  --> [Pengukur] Detak ke-5 pada 2.53 detik

=== UJI 3: Blocking CPU-bound + Pengukur ===
[CPU Bound] Mulai (BLOCKING CPU)...
[CPU Bound] Selesai.
  --> [Pengukur] Mulai mencatat waktu...
  --> [Pengukur] Detak ke-1 pada 0.51 detik
  --> [Pengukur] Detak ke-2 pada 1.02 detik
  --> [Pengukur] Detak ke-3 pada 1.53

In [ ]:
Analisis & Penjelasan:
Coroutine yang mengganggu pengukur: task_time_sleep() dan task_cpu_bound().

Mengapa terjadi demikian?

asyncio bekerja menggunakan single-threaded cooperative multitasking. Event loop hanya bisa berpindah antar-coroutine ketika sebuah coroutine memberikan kontrol kembali secara sukarela melalui kata kunci await.

time.sleep() memblokir seluruh thread utama secara synchronous, sehingga event loop tidak bisa berjalan untuk menjadwalkan coroutine pengukur.

task_cpu_bound() terus-menerus mengeksekusi instruksi CPU tanpa ada perintah await, sehingga menyita CPU dan menghentikan interupsi event loop hingga komputasi selesai.

In [ ]:
3. Producer-Multi-Consumer dengan Prioritas
Kita menggunakan asyncio.PriorityQueue agar item berprioritas tinggi (angka lebih kecil, contoh: 1) diproses lebih dahulu dibanding angka yang lebih besar (2, 3).

In [7]:
import asyncio
import random

async def producer(queue, count):
    print("--- Producer Mulai Memasukkan Item ---")
    for item_id in range(1, count + 1):
        prioritas = random.randint(1, 3) # Prioritas 1 (Tinggi) s.d 3 (Rendah)
        item_data = f"Tugas-{item_id}"
        
        # Masukkan tuple (prioritas, data) ke PriorityQueue
        await queue.put((prioritas, item_data))
        print(f"[Producer] Menambahkan: {item_data} (Prioritas: {prioritas})")
        await asyncio.sleep(0.1)

async def consumer(name, queue):
    while True:
        # Ambil item dari PriorityQueue
        prioritas, item_data = await queue.get()
        print(f"  [{name}] Memproses: {item_data} dengan Prioritas: {prioritas}")
        
        # Simulasi waktu pemrosesan
        await asyncio.sleep(0.5)
        queue.task_done()

async def main_eksperimen_3():
    queue = asyncio.PriorityQueue()
    num_consumers = 2
    num_items = 10

    # Jalankan producer
    await producer(queue, num_items)

    # Jalankan beberapa consumer
    consumers = [
        asyncio.create_task(consumer(f"Consumer-{i+1}", queue))
        for i in range(num_consumers)
    ]

    # Tunggu semua tugas di antrean selesai
    await queue.join()

    # Batalkan consumer task setelah tugas selesai
    for c in consumers:
        c.cancel()

await main_eksperimen_3()

--- Producer Mulai Memasukkan Item ---
[Producer] Menambahkan: Tugas-1 (Prioritas: 1)
[Producer] Menambahkan: Tugas-2 (Prioritas: 2)
[Producer] Menambahkan: Tugas-3 (Prioritas: 2)
[Producer] Menambahkan: Tugas-4 (Prioritas: 2)
[Producer] Menambahkan: Tugas-5 (Prioritas: 2)
[Producer] Menambahkan: Tugas-6 (Prioritas: 2)
[Producer] Menambahkan: Tugas-7 (Prioritas: 2)
[Producer] Menambahkan: Tugas-8 (Prioritas: 1)
[Producer] Menambahkan: Tugas-9 (Prioritas: 3)
[Producer] Menambahkan: Tugas-10 (Prioritas: 1)
  [Consumer-1] Memproses: Tugas-1 dengan Prioritas: 1
  [Consumer-2] Memproses: Tugas-10 dengan Prioritas: 1
  [Consumer-1] Memproses: Tugas-8 dengan Prioritas: 1
  [Consumer-2] Memproses: Tugas-2 dengan Prioritas: 2
  [Consumer-1] Memproses: Tugas-3 dengan Prioritas: 2
  [Consumer-2] Memproses: Tugas-4 dengan Prioritas: 2
  [Consumer-1] Memproses: Tugas-5 dengan Prioritas: 2
  [Consumer-2] Memproses: Tugas-6 dengan Prioritas: 2
  [Consumer-1] Memproses: Tugas-7 dengan Prioritas: 2
  [

In [ ]:
4.Refleksi Singkat

In [ ]:
asyncio merupakan pilihan paling efisien dan mudah dipahami untuk kasus I/O-bound karena beroperasi secara kooperatif pada single thread, yang membuat alur eksekusi jauh lebih prediktif dibandingkan threading yang rentan memicu race condition dan membutuhkan mekanisme lock yang rumit. Keunggulan ini makin terasa karena perpindahan kontrol antar-tugas selalu ditulis secara eksplisit menggunakan kata kunci await. Di sisi lain, asyncio juga jauh lebih ringan daripada multiprocessing yang terasa terlalu berat (heavyweight) untuk tugas I/O-bound akibat adanya overhead dari pembuatan proses baru serta alokasi memori yang terpisah. Pada akhirnya, sintaks async/await terbukti mampu memberikan kejelasan struktur kode layaknya sintaks sinkronus, namun tetap menghasilkan kinerja konkurensi yang sangat tinggi untuk menangani banyak operasi jaringan secara bersamaan.
Apakah Anda ingin saya membuat tabel perbandingan ringkas antara asyncio, threading, dan multiprocessing untuk mempertegas perbedaan karakteristik ketiganya?
AI can make mistakes, so double-check responses 
